In [ ]:
import pandas as pd
import numpy as np

In [ ]:
df_raw = pd.read_excel("Raw Data.xlsx", sheet_name="Normal Data", skiprows=2)

In [ ]:
df_raw.head()

In [ ]:
# Format cols

df_raw["sex"] = df_raw["Sex"].str.upper()
df_raw["age"] = df_raw["Unnamed: 1"]
df_raw["patient_id"] = df_raw["Unnamed: 0"]
df_raw["height"] = df_raw["Unnamed: 3"]
df_raw["weight"] = df_raw["Unnamed: 4"]

In [ ]:
df_patient_demographics = df_raw[["patient_id", "sex", "age", "height", "weight"]]
df_patient_demographics.head()

In [ ]:
df_patient_demographics.patient_id.nunique()

In [ ]:
df_patient_demographics.to_csv(
    "data/demographics.csv", index=False
)  # Save patient demographics

In [ ]:
# Bring in measured data
df = pd.read_excel("Raw Data.xlsx", sheet_name="Normal Data", header=[0, 1, 2])

In [ ]:
# Create long form dataset.
# New patient records per side and muscle group

measures_summary_dict = {}
for x in ["CMCT (msec)", "MEP Latency (msec)", "MEP Amplitude (mV)"]:
    long_format_per_measure = []
    for y in [
        "Median (APB",
        "Median PT",
        "Ulnar ADM",
        "Ulnar FDI",
        "Peroneal TA",
        "Tibial AH",
    ]:
        for z in ["Left", "Right"]:
            df_updated = df_patient_demographics.copy()
            df_updated["measure"] = x
            df_updated["muscle"] = y
            df_updated["side"] = z
            df_updated["limb"] = np.where(
                df_updated["muscle"].isin(
                    ["Median (APB", "Median PT", "Ulnar ADM", "Ulnar FDI"]
                ),
                "upper",
                "lower",
            )
            df_updated["value"] = df[x][y][z]
            long_format_per_measure.append(df_updated)
    pd.concat(long_format_per_measure).to_csv(f"data/{x}_all.csv", index=False)

In [ ]:
# Data missingness
# Grouped by side, muscle group, sex and overall

missing_data_list = []
for x in ["CMCT (msec)", "MEP Latency (msec)", "MEP Amplitude (mV)"]:

    data = pd.read_csv(f"data/{x}_all.csv")

    display(data.isna().sum())

    data["missing_reading"] = data["value"].isna()

    by_side = data.groupby("side")["missing_reading"].agg(["sum", "mean"]).reset_index()
    by_side["perc"] = 100 * by_side["mean"]
    by_side.rename(columns={"side": "level"}, inplace=True)
    by_side["grouped_by"] = "side"

    by_muscle = (
        data.groupby("muscle")["missing_reading"].agg(["sum", "mean"]).reset_index()
    )
    by_muscle["perc"] = 100 * by_muscle["mean"]
    by_muscle.rename(columns={"muscle": "level"}, inplace=True)
    by_muscle["grouped_by"] = "muscle"

    by_sex = data.groupby("sex")["missing_reading"].agg(["sum", "mean"]).reset_index()
    by_sex["perc"] = 100 * by_sex["mean"]
    by_sex.rename(columns={"sex": "level"}, inplace=True)
    by_sex["grouped_by"] = "sex"

    overall = pd.DataFrame(
        {
            "grouped_by": ["all"],
            "level": ["all"],
            "sum": [data["missing_reading"].sum()],
            "mean": [data["missing_reading"].mean()],
        }
    )

    missing_summary_df = pd.concat([by_side, by_muscle, by_sex, overall])[
        ["grouped_by", "level", "sum", "mean"]
    ]
    missing_summary_df["measure"] = x
    missing_data_list.append(missing_summary_df)

pd.concat(missing_data_list).to_csv("results/missing_summary.csv", index=False)

In [ ]:
# Re-save data with missing values dropped

measures_summary_dict = {}
for x in ["CMCT (msec)", "MEP Latency (msec)", "MEP Amplitude (mV)"]:

    data = pd.read_csv(f"data/{x}_all.csv")
    data = data.dropna(subset=["value"])
    display(data.isna().sum())
    data.to_csv(f"data/{x}.csv", index=False)